# Activation Maps Visualization

This notebook provides tools to visualize and compare activation maps generated by `exp_actmaps.py`.

In [ ]:
import torch
import matplotlib.pyplot as plt
import os
import math
import numpy as np

def build_actmap_path(model_name):


    
data = torch.load(data_path, map_location='cpu')



In [ ]:
# Function to retrieve and plot activation maps for a single model
def visualize_model_actmaps(model_path_stem):
    """
    Retrieves activation maps for a model and prints them.

    Args:
        model_path_stem (str): Path to the saved figure/data without extension.
                               Can be a directory as well.
                               Example: 'save/ResNet18/figures/actmaps/crossentropy...'
    """
    data_path = find_pt_file(model_path_stem)

    if not data_path or not os.path.exists(data_path):
        print(f"Error: .pt file not found for {model_path_stem}")
        return

    print(f"Loading data from {data_path}...")
    try:
        act_matrices = torch.load(data_path, map_location='cpu')
    except Exception as e:
        print(f"Error loading .pt file: {e}")
        return

    class_names = list(act_matrices.keys())
    num_classes = len(class_names)
    
    # Determine grid layout
    cols = 5
    rows = math.ceil(num_classes / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = axes.flatten()

    # Find global min/max for consistent types
    # We can either normalize per image or globally. Usually per image is fine,
    # but consistent scale helps comparison. Let's do per image for single view.

    for i, class_name in enumerate(class_names):
        heatmap = act_matrices[class_name]
        ax = axes[i]
        
        v = heatmap.abs().max().item()
        
        im = ax.imshow(heatmap, cmap='bwr', interpolation='nearest', vmin=-v, vmax=v)
        ax.set_title(class_name)
        ax.axis('off')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # Turn off unused axes
    for i in range(num_classes, len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

def compare_models_actmaps(model_path_stems, labels=None):
    """
    Compares activation maps across multiple models that vary by rho.

    Args:
        model_path_stems (list): List of paths to model data files (stems) or directories.
        labels (list, optional): List of labels for each model (e.g. rho values).
                                 If None, uses the last part of the path.
    """
    loaded_data = []
    valid_labels = []

    for i, path in enumerate(model_path_stems):
        data_path = find_pt_file(path)
        
        if data_path and os.path.exists(data_path):
            try:
                data = torch.load(data_path, map_location='cpu')
                loaded_data.append(data)
                if labels:
                    valid_labels.append(labels[i])
                else:
                    # Try to extract a meaningful label, e.g., the filename
                    # Use everything after 'rho' if presents, or just the basename
                    basename = os.path.basename(path)
                    if 'rho' in basename:
                         # A heuristic to extract rho value
                         # E.g. ..._0.04rho_... -> 0.04rho
                         parts = basename.split('_')
                         rho_part = [p for p in parts if 'rho' in p and p != 'rho']
                         if rho_part:
                             valid_labels.append(rho_part[0])
                         else:
                             valid_labels.append(basename)
                    else:
                        valid_labels.append(basename)
            except Exception as e:
                print(f"Error loading {data_path}: {e}")
        else:
            print(f"Warning: File not found for {path}")

    if not loaded_data:
        print("No data loaded.")
        return

    # Assume all loaded data have the same classes
    class_names = list(loaded_data[0].keys())
    num_classes = len(class_names)
    num_models = len(loaded_data)

    # Create a big grid: Rows = Classes, Cols = Models
    # Ensure axes is always 2D array
    fig, axes = plt.subplots(num_classes, num_models, figsize=(3 * num_models, 3 * num_classes), squeeze=False)
    
    for r, class_name in enumerate(class_names):
        for c in range(num_models):
            ax = axes[r, c]
            data = loaded_data[c]
            
            if class_name in data:
                heatmap = data[class_name]
                v = heatmap.abs().max().item()
                im = ax.imshow(heatmap, cmap='bwr', interpolation='nearest', vmin=-v, vmax=v)
                
                if r == 0:
                    ax.set_title(valid_labels[c], fontsize=10)
                if c == 0:
                    ax.set_ylabel(class_name, fontsize=12)
                
                # Remove ticks for cleaner look
                ax.set_xticks([])
                ax.set_yticks([])
                # Optional: Add colorbar per plot or global colorbar
            else:
                ax.text(0.5, 0.5, "N/A", ha='center', va='center')
                ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# Example Usage:
# 1. Visualize a single model
# model_path = "save/ResNet18/figures/actmaps/crossentropy_wstopo_256embdims_0.04rho_50epochs_512bsz_2nwork_0.002lr_0.5dropout"
# visualize_model_actmaps(model_path)

# 2. Compare models with different rho values
# model_base = "save/ResNet18/figures/actmaps/"
# rho_models = [
#     model_base + "crossentropy_wstopo_256embdims_0.0rho_50epochs_512bsz_2nwork_0.002lr_0.5dropout",
#     model_base + "crossentropy_wstopo_256embdims_0.04rho_50epochs_512bsz_2nwork_0.002lr_0.5dropout",
#     model_base + "crossentropy_wstopo_256embdims_0.2rho_50epochs_512bsz_2nwork_0.002lr_0.5dropout"
# ]
# # labels = ["rho=0.0", "rho=0.04", "rho=0.2"]
# compare_models_actmaps(rho_models)